# Итоговый проект — Вариант RAG-02
# Ассистент по документации DevOps/Python

**Модуль 07 · AI HUB · Трек Б (RAG-чат)**

---

## Задача

Построить RAG-ассистента по технической документации. Разработчик задаёт how-to вопросы про Docker, FastAPI, scikit-learn и получает ответ со ссылкой на конкретную страницу документации.

**Корпус:** ≥ 15 страниц из ≥ 2 источников.

```bash
pip install requests beautifulsoup4
mkdir -p data/docs

# scikit-learn User Guide (RST из GitHub)
BASE="https://raw.githubusercontent.com/scikit-learn/scikit-learn/main/doc/modules"
for page in preprocessing cross_validation pipeline model_evaluation ensemble; do
  wget -q -O "data/docs/sklearn_${page}.rst" "${BASE}/${page}.rst"
done

# Docker docs (MD из GitHub)
BASE="https://raw.githubusercontent.com/docker/docs/main/content/get-started"
for page in introduction what-is-docker containerize; do
  wget -q -O "data/docs/docker_${page}.md" "${BASE}/${page}/index.md"
done
```

FastAPI страницы загружаются скриптом (см. Задание 1).

**Стек:** wget/requests → HTML/RST/MD → chunking → FAISS → FastAPI `/ask` → Streamlit → Docker

**Целевые метрики:** Recall@3 ≥ 0.65 на ≥ 20 how-to вопросах

---

## Задание 1 — Загрузка FastAPI документации

Напишите скрипт, который скачивает 6 страниц FastAPI tutorial через `requests.get()` и сохраняет в `data/docs/`:

- `fastapi_intro` — https://fastapi.tiangolo.com/
- `fastapi_path_params` — .../tutorial/path-params/
- `fastapi_body` — .../tutorial/body/
- `fastapi_response` — .../tutorial/response-model/
- `fastapi_dependencies` — .../tutorial/dependencies/
- `fastapi_testing` — .../tutorial/testing/

Удалите HTML-теги через `re.sub(r'<[^>]+>', ' ', html)` перед сохранением.

> ⚠️ Если сохранить HTML как есть — теги и CSS-классы попадут в словарь TF-IDF и испортят поиск. Всегда очищайте HTML перед индексацией.

---

## Задание 2 — Очистка разных форматов

Напишите функцию `load_doc(path)`, которая читает файл и применяет очистку в зависимости от расширения:

- `.rst` — RST-файлы scikit-learn содержат директивы вида `:class:\`SomeClass\``, `.. code-block::`, `.. note::`. Уберите их регуляркой: `re.sub(r':\w+:`([^`]*)`', r'\1', text)`
- `.md` — Markdown читается как есть, лишняя разметка (`##`, `**`, `` ` ``) не мешает эмбеддингам
- `.txt` — без дополнительной обработки

Выведите количество слов для каждого документа. Убедитесь, что ни один не пустой.

---

## Задание 3 — Chunking

**Минимум:** простое разбиение по 400-500 слов с overlap 50.

**На хорошую оценку (Good):** реализуйте code-aware chunking:
- Разбивайте по заголовкам `##` / `###` в MD-файлах
- **Никогда не разрывайте блоки ` ``` `** — отслеживайте флаг `is_in_code_block`
- Если секция после заголовка > max_words — разбейте дальше по словам

Проверьте: ни один чанк не содержит только открывающий или только закрывающий ` ``` `.

---

## Задание 4 — Индексирование в FAISS

1. Переберите все файлы в `data/docs/` с расширениями `.txt`, `.md`, `.rst`
2. Для каждого: загрузите через `load_doc()` → разбейте на чанки → закодируйте с префиксом `"passage: "`
3. `source` = имя файла (`fastapi_body.txt`, `sklearn_pipeline.rst`)
4. Создайте `faiss.IndexFlatIP(DIM)`, добавьте векторы, сохраните индекс и метаданные

---

## Задание 5 — Функция `ask()` и тестирование

Реализуйте `search(question, top_k=6)` и `ask(question, top_k=6)`. Для технических вопросов рекомендуется `top_k=6` — больше контекста помогает LLM.

System prompt: ассистент по документации, отвечает только из контекста, указывает источник.

Проверьте вручную 3–5 how-to вопросов:

---

## Задание 6 — Recall@3

Составьте ≥ 20 тест-пар `(вопрос, имя_файла)` с how-to вопросами. Пишите вопросы так, как спросил бы реальный разработчик — без точных фраз из документации.

Реализуйте `recall_at_k(pairs, k)` и запустите при k = 1, 3, 5.

Цель: Recall@3 ≥ 0.65. Если ниже — попробуйте code-aware chunking или увеличьте `top_k`.

**Результаты:**
- Recall@1 = *(запишите)*
- Recall@3 = *(запишите)*
- Recall@5 = *(запишите)*

**Какие вопросы провалились и почему?** *(запишите)*

---

## Задание 7 — FastAPI-сервис и Docker

Создайте `app/main.py`:

**`GET /health`** → `{"status": "ok", "chunks_indexed": N}`

**`POST /ask`** принимает `{"question": "...", "top_k": 6}`, возвращает `{"answer": "...", "sources": [...], "hits": [...]}`

Напишите тесты (≥ 5): health, вопрос по FastAPI → источник `fastapi_*`, вопрос по sklearn → источник `sklearn_*`, out-of-scope, пустой вопрос.

Создайте `docker-compose.yml` с сервисами `api` (8000) и `ui` (8501).

---

## Чеклист сдачи

### Обязательный минимум (Pass)
- [ ] ≥ 15 документов из ≥ 2 источников, HTML/RST очищен
- [ ] FAISS `IndexFlatIP` + `faiss_meta.pkl` сохранены
- [ ] FastAPI: `GET /health` + `POST /ask` (answer + sources + hits с text)
- [ ] Streamlit: чат, блок источников с текстом фрагмента
- [ ] Recall@3 ≥ 0.65 на ≥ 20 вопросах
- [ ] pytest: ≥ 5 тестов
- [ ] `docker-compose.yml`

### На хорошую оценку (Good)
- [ ] Code-aware chunking: не разрывать ``` блоки
- [ ] Streamlit: `source` — кликабельная ссылка на онлайн-документацию
- [ ] Recall@3 ≥ 0.75

### Дополнительно (Excellent)
- [ ] `update_docs.py` — скачать актуальную версию и пересобрать индекс
- [ ] Sidebar: фильтр по источнику (Docker / FastAPI / sklearn)
- [ ] Code highlighting в Streamlit для блоков с кодом